<a href="https://colab.research.google.com/github/2303A51528/reinforcement_learning_1528/blob/main/practical_12_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Assignment 12: Value Iteration in a Deterministic 2×2 Grid World
This notebook implements the Value Iteration algorithm for a deterministic 2x2 Grid World to obtain optimal state values and the optimal policy.

In [1]:
import pandas as pd
import numpy as np

# 1. Define the MDP environment parameters
states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']
actions = ['Up', 'Down', 'Left', 'Right']

gamma = 0.9
reward_normal = -1.0
reward_goal = 100.0

# Define deterministic transitions and rewards: transitions[state][action] = (next_state, reward)
transitions = {
    'S0': {
        'Up': ('S0', reward_normal),
        'Down': ('S1', reward_normal),
        'Left': ('S0', reward_normal),
        'Right': ('S0', reward_normal)
    },
    'S1': {
        'Up': ('S0', reward_normal),
        'Down': ('S1', reward_normal),
        'Left': ('S1', reward_normal),
        'Right': ('G', reward_goal)
    }
}

# Print MDP transition information
print("--- MDP Transitions and Rewards Information ---")
for state in non_terminal_states:
    for action in actions:
        next_state, reward = transitions[state][action]
        print(f"State: {state} | Action: {action:<5} -> Next State: {next_state:<2} | Reward: {reward}")

--- MDP Transitions and Rewards Information ---
State: S0 | Action: Up    -> Next State: S0 | Reward: -1.0
State: S0 | Action: Down  -> Next State: S1 | Reward: -1.0
State: S0 | Action: Left  -> Next State: S0 | Reward: -1.0
State: S0 | Action: Right -> Next State: S0 | Reward: -1.0
State: S1 | Action: Up    -> Next State: S0 | Reward: -1.0
State: S1 | Action: Down  -> Next State: S1 | Reward: -1.0
State: S1 | Action: Left  -> Next State: S1 | Reward: -1.0
State: S1 | Action: Right -> Next State: G  | Reward: 100.0


In [2]:
# 2. Value Iteration Algorithm
theta = 1e-6  # Convergence threshold
V = {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

iteration_history = []
iteration_history.append({'Iteration': 0, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'V(G)': V['G']})

iteration = 1
while True:
    delta = 0
    V_new = V.copy()

    for s in non_terminal_states:
        v = V[s]
        q_values = []
        for a in actions:
            next_state, reward = transitions[s][a]
            q_val = reward + gamma * V[next_state]
            q_values.append(q_val)

        V_new[s] = max(q_values)
        delta = max(delta, abs(V_new[s] - v))

    V = V_new
    iteration_history.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'V(G)': V['G']})

    if delta < theta:
        break
    iteration += 1

# Display Iteration-wise state-value table
df_iterations = pd.DataFrame(iteration_history)
print("\n--- Iteration-wise State-Value Table ---")
display(df_iterations)


--- Iteration-wise State-Value Table ---


,Iteration,V(S0),V(S1),V(G)
0,0,0.0,0.0,0.0
1,1,-1.0,100.0,0.0
2,2,89.0,100.0,0.0
3,3,89.0,100.0,0.0


In [3]:
# 3. Calculate final action-values (Q*) and determine optimal policy
q_table = []
optimal_policy = {}

for s in non_terminal_states:
    best_action = None
    best_q = -float('inf')
    for a in actions:
        next_state, reward = transitions[s][a]
        q_val = reward + gamma * V[next_state]
        q_table.append({'State': s, 'Action': a, 'Next State': next_state, 'Reward': reward, 'Q*(s,a)': round(q_val, 2)})

        if q_val > best_q:
            best_q = q_val
            best_action = a
    optimal_policy[s] = (best_action, round(best_q, 2))

# Display final action-value table Q*(s, a)
df_q = pd.DataFrame(q_table)
print("--- Final Action-Value Table Q*(s, a) ---")
display(df_q)

--- Final Action-Value Table Q*(s, a) ---


,State,Action,Next State,Reward,"Q*(s,a)"
0,S0,Up,S0,-1.0,79.1
1,S0,Down,S1,-1.0,89.0
2,S0,Left,S0,-1.0,79.1
3,S0,Right,S0,-1.0,79.1
4,S1,Up,S0,-1.0,79.1
5,S1,Down,S1,-1.0,89.0
6,S1,Left,S1,-1.0,89.0
7,S1,Right,G,100.0,100.0


In [4]:
# 4. Display Final Optimal Results
print("--- Final Optimal State Values V*(s) ---")
for s in states:
    print(f"V*({s}) = {V[s]:.1f}")

print("\n--- Optimal Policy ---")
for s in non_terminal_states:
    action, val = optimal_policy[s]
    print(f"State {s}: Best Action is '{action}' with Value {val:.1f}")

print("\nOptimal Path: S0 -> S1 -> G")

--- Final Optimal State Values V*(s) ---
V*(S0) = 89.0
V*(S1) = 100.0
V*(G) = 0.0

--- Optimal Policy ---
State S0: Best Action is 'Down' with Value 89.0
State S1: Best Action is 'Right' with Value 100.0

Optimal Path: S0 -> S1 -> G
